# FounderAI Colab Free V1

Notebook standalone: dataset + scripts de training inclus. Une fois le notebook ouvert dans Colab, tu peux faire `Run all`.

Avant de lancer:
- Ouvre `Runtime > Change runtime type`
- Choisis `T4 GPU` si disponible
- Ensuite clique `Run all`


In [ ]:
import os
from pathlib import Path

RUN_ROOT = Path('/content/founderai-colab-v1')
TRAINING_DIR = RUN_ROOT / 'training_data'
OUTPUT_DIR = RUN_ROOT / 'lora_adapter'
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Run root:', RUN_ROOT)


In [ ]:
!pip install -q transformers>=4.51.0 peft>=0.10.0 accelerate>=0.28.0 bitsandbytes>=0.43.0 datasets>=2.18.0 trl>=0.8.0 torch>=2.2.0 sentencepiece>=0.2.0 protobuf>=4.25.0


In [ ]:
!nvidia-smi


In [ ]:
import base64
import gzip
from pathlib import Path

DATASET_B64 = '''...'''
dataset_path = Path('/content/founderai-colab-v1/training_data/teranga_merged.jsonl')
dataset_path.write_bytes(gzip.decompress(base64.b64decode(DATASET_B64)))
print('Dataset written to:', dataset_path)
print('Dataset size bytes:', dataset_path.stat().st_size)


In [ ]:
from pathlib import Path

finetune_utils_path = Path('/content/founderai-colab-v1/training_data/finetune_utils.py')
train_script_path = Path('/content/founderai-colab-v1/training_data/train_qwen3_lora_colab.py')
finetune_utils_path.write_text('"""Utility helpers shared by training and evaluation scripts."""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom datasets import Dataset\n\n\nVALID_SPLITS = {"train", "validation", "test"}\n\nDEFAULT_CHAT_CHARS_PER_TOKEN = 4.0\n\n\ndef load_jsonl_records(path: str | Path) -> list[dict]:\n    file_path = Path(path)\n    records: list[dict] = []\n    with file_path.open("r", encoding="utf-8") as handle:\n        for line in handle:\n            line = line.strip()\n            if not line:\n                continue\n            records.append(json.loads(line))\n    return records\n\n\ndef normalize_split(split: str | None) -> str:\n    candidate = (split or "").strip().lower()\n    return candidate if candidate in VALID_SPLITS else "train"\n\n\ndef split_records(records: list[dict]) -> dict[str, list[dict]]:\n    buckets = {"train": [], "validation": [], "test": []}\n    for record in records:\n        buckets[normalize_split(record.get("split"))].append(record)\n    return buckets\n\n\ndef records_to_messages_dataset(records: list[dict]) -> Dataset:\n    return Dataset.from_dict({"messages": [record["messages"] for record in records]})\n\n\ndef render_messages_as_text(messages: list[dict]) -> str:\n    return "\\n".join(message["content"] for message in messages)\n\n\ndef apply_chat_template(example: dict, tokenizer) -> dict:\n    text = tokenizer.apply_chat_template(\n        example["messages"],\n        tokenize=False,\n        add_generation_prompt=False,\n    )\n    return {"text": text}\n\n\ndef tokenize_example(example: dict, tokenizer, max_seq_length: int) -> dict:\n    tokenized = tokenizer(\n        example["text"],\n        truncation=True,\n        max_length=max_seq_length,\n        padding="max_length",\n    )\n    labels = tokenized["input_ids"].copy()\n    pad_token_id = tokenizer.pad_token_id\n    if pad_token_id is not None:\n        labels = [token if token != pad_token_id else -100 for token in labels]\n    tokenized["labels"] = labels\n    return tokenized\n\n\ndef prepare_split_dataset(records: list[dict], tokenizer, max_seq_length: int) -> Dataset:\n    dataset = records_to_messages_dataset(records)\n    dataset = dataset.map(\n        lambda row: apply_chat_template(row, tokenizer),\n        remove_columns=["messages"],\n    )\n    dataset = dataset.map(\n        lambda row: tokenize_example(row, tokenizer, max_seq_length),\n        remove_columns=["text"],\n    )\n    return dataset\n\n\n@dataclass\nclass RecordProfile:\n    record_id: str\n    split: str\n    language: str\n    task_type: str\n    module: str\n    source_dataset: str\n    char_length: int\n    estimated_tokens: int\n    difficulty_score: int\n\n\ndef estimate_token_count_from_text(text: str, chars_per_token: float = DEFAULT_CHAT_CHARS_PER_TOKEN) -> int:\n    return max(1, math.ceil(len(text) / chars_per_token))\n\n\ndef score_record_difficulty(record: dict, estimated_tokens: int) -> int:\n    task_type = record.get("task_type", "teranga_native")\n    task_weight = {\n        "problem_statement_generation": 1,\n        "problem_statement_rewrite": 1,\n        "interview_script": 1,\n        "icp_creation": 1,\n        "problem_validation_plan": 2,\n        "interview_debrief": 2,\n        "icp_critique": 2,\n        "market_sizing": 2,\n        "roi_modeling": 2,\n        "bmc_draft": 3,\n        "assumption_mapping": 3,\n        "user_journey_map": 3,\n        "teranga_native": 4,\n    }\n    length_weight = 1\n    if estimated_tokens >= 550:\n        length_weight = 4\n    elif estimated_tokens >= 420:\n        length_weight = 3\n    elif estimated_tokens >= 300:\n        length_weight = 2\n    return task_weight.get(task_type, 3) * 10 + length_weight\n\n\ndef build_record_profiles(records: list[dict]) -> list[RecordProfile]:\n    profiles: list[RecordProfile] = []\n    for index, record in enumerate(records, start=1):\n        text = render_messages_as_text(record["messages"])\n        estimated_tokens = estimate_token_count_from_text(text)\n        profiles.append(\n            RecordProfile(\n                record_id=str(record.get("id") or f"record_{index:04d}"),\n                split=normalize_split(record.get("split")),\n                language=record.get("language", "mixed"),\n                task_type=record.get("task_type", "teranga_native"),\n                module=record.get("teranga_module", "unknown"),\n                source_dataset=record.get("source_dataset", "unknown"),\n                char_length=len(text),\n                estimated_tokens=estimated_tokens,\n                difficulty_score=score_record_difficulty(record, estimated_tokens),\n            )\n        )\n    return profiles\n\n\ndef dataset_stats(records: list[dict]) -> dict:\n    by_split = Counter(normalize_split(record.get("split")) for record in records)\n    by_language = Counter(record.get("language", "mixed") for record in records)\n    by_source = Counter(record.get("source_dataset", "unknown") for record in records)\n    profiles = build_record_profiles(records)\n    estimated_tokens = sorted(profile.estimated_tokens for profile in profiles)\n\n    def percentile(p: float) -> int:\n        if not estimated_tokens:\n            return 0\n        idx = min(len(estimated_tokens) - 1, max(0, round((len(estimated_tokens) - 1) * p)))\n        return estimated_tokens[idx]\n\n    return {\n        "total_records": len(records),\n        "by_split": dict(sorted(by_split.items())),\n        "by_language": dict(sorted(by_language.items())),\n        "by_source_dataset": dict(sorted(by_source.items())),\n        "estimated_token_summary": {\n            "min": estimated_tokens[0] if estimated_tokens else 0,\n            "p50": percentile(0.50),\n            "p75": percentile(0.75),\n            "p90": percentile(0.90),\n            "p95": percentile(0.95),\n            "max": estimated_tokens[-1] if estimated_tokens else 0,\n        },\n    }\n\n\ndef add_perplexity(metrics: dict) -> dict:\n    enriched = dict(metrics)\n    for key in ("eval_loss", "validation_loss", "test_loss", "train_loss"):\n        if key in enriched:\n            try:\n                enriched[key.replace("loss", "perplexity")] = math.exp(float(enriched[key]))\n            except OverflowError:\n                enriched[key.replace("loss", "perplexity")] = float("inf")\n    return enriched\n', encoding='utf-8')
train_script_path.write_text('"""Colab-friendly QLoRA fine-tuning entrypoint for FounderAI.\n\nThis script is tuned for Google Colab free/limited GPU sessions:\n- portable paths (no Windows-only defaults)\n- 4-bit QLoRA by default\n- shorter context windows to fit smaller GPUs\n- resumable checkpoints\n- Colab-local output paths by default\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Optional\n\nimport torch\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\nfrom transformers import (\n    AutoModelForCausalLM,\n    AutoTokenizer,\n    BitsAndBytesConfig,\n    Trainer,\n    TrainingArguments,\n    default_data_collator,\n)\nfrom transformers.trainer_utils import get_last_checkpoint\n\ntry:\n    from .finetune_utils import add_perplexity, dataset_stats, load_jsonl_records, prepare_split_dataset, split_records\nexcept ImportError:\n    from finetune_utils import add_perplexity, dataset_stats, load_jsonl_records, prepare_split_dataset, split_records\n\n\ndef _repo_root() -> Path:\n    return Path(__file__).resolve().parents[1]\n\n\ndef _env_path(name: str, default: Path) -> Path:\n    return Path(os.getenv(name, str(default))).expanduser()\n\n\ndef _env_int(name: str, default: int) -> int:\n    raw = os.getenv(name)\n    if raw is None or raw == "":\n        return default\n    return int(raw)\n\n\ndef _env_float(name: str, default: float) -> float:\n    raw = os.getenv(name)\n    if raw is None or raw == "":\n        return default\n    return float(raw)\n\n\n@dataclass\nclass ColabTrainingConfig:\n    base_model_id: str = field(default_factory=lambda: os.getenv("FOUNDER_AI_COLAB_BASE_MODEL", "Qwen/Qwen3-4B"))\n    data_path: Path = field(default_factory=lambda: _env_path("FOUNDER_AI_COLAB_DATA_PATH", _repo_root() / "training_data" / "teranga_merged.jsonl"))\n    output_dir: Path = field(default_factory=lambda: _env_path("FOUNDER_AI_COLAB_OUTPUT_DIR", Path("/content/founderai-colab-v1/lora_adapter")))\n    metrics_path: Path = field(default_factory=lambda: _env_path("FOUNDER_AI_COLAB_METRICS_PATH", Path("/content/founderai-colab-v1/lora_adapter/training_metrics.json")))\n    sample_limit: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_SAMPLE_LIMIT", 0))\n\n    lora_r: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_LORA_R", 8))\n    lora_alpha: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_LORA_ALPHA", 16))\n    lora_dropout: float = field(default_factory=lambda: _env_float("FOUNDER_AI_COLAB_LORA_DROPOUT", 0.05))\n    lora_target_modules: list[str] = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"])\n\n    num_train_epochs: float = field(default_factory=lambda: _env_float("FOUNDER_AI_COLAB_EPOCHS", 1.0))\n    per_device_train_batch_size: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_TRAIN_BATCH", 1))\n    per_device_eval_batch_size: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_EVAL_BATCH", 1))\n    gradient_accumulation_steps: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_GRAD_ACCUM", 8))\n    learning_rate: float = field(default_factory=lambda: _env_float("FOUNDER_AI_COLAB_LR", 2e-4))\n    warmup_ratio: float = field(default_factory=lambda: _env_float("FOUNDER_AI_COLAB_WARMUP_RATIO", 0.03))\n    max_seq_length: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_MAX_SEQ_LENGTH", 512))\n    logging_steps: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_LOGGING_STEPS", 5))\n    save_steps: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_SAVE_STEPS", 25))\n    eval_steps: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_EVAL_STEPS", 25))\n    save_total_limit: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_SAVE_TOTAL_LIMIT", 2))\n\n    use_4bit: bool = field(default_factory=lambda: os.getenv("FOUNDER_AI_COLAB_USE_4BIT", "true").lower() == "true")\n    use_8bit: bool = field(default_factory=lambda: os.getenv("FOUNDER_AI_COLAB_USE_8BIT", "false").lower() == "true")\n    gradient_checkpointing: bool = field(default_factory=lambda: os.getenv("FOUNDER_AI_COLAB_GRADIENT_CHECKPOINTING", "true").lower() == "true")\n    seed: int = field(default_factory=lambda: _env_int("FOUNDER_AI_COLAB_SEED", 42))\n\n\ndef create_lora_config(config: ColabTrainingConfig) -> LoraConfig:\n    return LoraConfig(\n        r=config.lora_r,\n        lora_alpha=config.lora_alpha,\n        lora_dropout=config.lora_dropout,\n        target_modules=config.lora_target_modules,\n        bias="none",\n        task_type="CAUSAL_LM",\n    )\n\n\ndef create_quantization_config(config: ColabTrainingConfig) -> Optional[BitsAndBytesConfig]:\n    if config.use_4bit:\n        compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16\n        return BitsAndBytesConfig(\n            load_in_4bit=True,\n            bnb_4bit_quant_type="nf4",\n            bnb_4bit_compute_dtype=compute_dtype,\n            bnb_4bit_use_double_quant=True,\n        )\n    if config.use_8bit:\n        return BitsAndBytesConfig(load_in_8bit=True)\n    return None\n\n\ndef load_model_and_tokenizer(config: ColabTrainingConfig):\n    tokenizer = AutoTokenizer.from_pretrained(config.base_model_id, trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token\n    tokenizer.padding_side = "right"\n\n    quant_config = create_quantization_config(config)\n    use_cuda = torch.cuda.is_available()\n    default_dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else (torch.float16 if use_cuda else torch.float32)\n\n    model_kwargs = {\n        "trust_remote_code": True,\n        "torch_dtype": default_dtype,\n    }\n    if quant_config is not None and use_cuda:\n        model_kwargs["quantization_config"] = quant_config\n        model_kwargs["device_map"] = "auto"\n\n    model = AutoModelForCausalLM.from_pretrained(config.base_model_id, **model_kwargs)\n    model.config.use_cache = False\n\n    if quant_config is not None:\n        model = prepare_model_for_kbit_training(model)\n\n    if config.gradient_checkpointing:\n        model.gradient_checkpointing_enable()\n        model.enable_input_require_grads()\n\n    model = get_peft_model(model, create_lora_config(config))\n    return model, tokenizer\n\n\ndef build_datasets(config: ColabTrainingConfig, tokenizer):\n    records = load_jsonl_records(config.data_path)\n    split_map = split_records(records)\n\n    if config.sample_limit > 0:\n        split_map["train"] = split_map["train"][: config.sample_limit]\n\n    missing = [name for name in ("train", "validation", "test") if not split_map[name]]\n    if missing:\n        raise ValueError(f"Missing dataset splits in {config.data_path}: {\', \'.join(missing)}")\n\n    train_dataset = prepare_split_dataset(split_map["train"], tokenizer, config.max_seq_length)\n    validation_dataset = prepare_split_dataset(split_map["validation"], tokenizer, config.max_seq_length)\n    test_dataset = prepare_split_dataset(split_map["test"], tokenizer, config.max_seq_length)\n    return split_map, train_dataset, validation_dataset, test_dataset\n\n\ndef write_metrics(path: Path, metrics: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef gpu_summary() -> dict:\n    if not torch.cuda.is_available():\n        return {"cuda": False}\n    props = torch.cuda.get_device_properties(0)\n    return {\n        "cuda": True,\n        "name": props.name,\n        "total_memory_gb": round(props.total_memory / (1024 ** 3), 2),\n        "bf16_supported": bool(torch.cuda.is_bf16_supported()),\n    }\n\n\ndef main() -> None:\n    config = ColabTrainingConfig()\n    config.output_dir.mkdir(parents=True, exist_ok=True)\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\n            "No GPU detected. In Colab, switch to Runtime > Change runtime type > T4 GPU before launching training."\n        )\n\n    print(json.dumps({"gpu": gpu_summary(), "config": {**asdict(config), "data_path": str(config.data_path), "output_dir": str(config.output_dir), "metrics_path": str(config.metrics_path)}}, ensure_ascii=False, indent=2))\n\n    model, tokenizer = load_model_and_tokenizer(config)\n    model.print_trainable_parameters()\n\n    split_map, train_dataset, validation_dataset, test_dataset = build_datasets(config, tokenizer)\n    merged_records = split_map["train"] + split_map["validation"] + split_map["test"]\n    print(json.dumps(dataset_stats(merged_records), ensure_ascii=False, indent=2))\n    print(\n        f"Prepared datasets -> train: {len(train_dataset)}, "\n        f"validation: {len(validation_dataset)}, test: {len(test_dataset)}"\n    )\n\n    training_args = TrainingArguments(\n        output_dir=str(config.output_dir),\n        do_train=True,\n        do_eval=True,\n        eval_strategy="steps",\n        eval_steps=config.eval_steps,\n        save_strategy="steps",\n        save_steps=config.save_steps,\n        load_best_model_at_end=True,\n        metric_for_best_model="eval_loss",\n        greater_is_better=False,\n        num_train_epochs=config.num_train_epochs,\n        per_device_train_batch_size=config.per_device_train_batch_size,\n        per_device_eval_batch_size=config.per_device_eval_batch_size,\n        gradient_accumulation_steps=config.gradient_accumulation_steps,\n        learning_rate=config.learning_rate,\n        warmup_ratio=config.warmup_ratio,\n        logging_steps=config.logging_steps,\n        save_total_limit=config.save_total_limit,\n        fp16=bool(torch.cuda.is_available() and not torch.cuda.is_bf16_supported()),\n        bf16=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),\n        seed=config.seed,\n        report_to="none",\n        optim="paged_adamw_8bit" if (config.use_4bit or config.use_8bit) else "adamw_torch",\n        lr_scheduler_type="cosine",\n        max_grad_norm=1.0,\n        weight_decay=0.01,\n        remove_unused_columns=False,\n        dataloader_pin_memory=True,\n        gradient_checkpointing=config.gradient_checkpointing,\n    )\n\n    trainer = Trainer(\n        model=model,\n        args=training_args,\n        train_dataset=train_dataset,\n        eval_dataset=validation_dataset,\n        tokenizer=tokenizer,\n        data_collator=default_data_collator,\n    )\n\n    last_checkpoint = get_last_checkpoint(str(config.output_dir))\n    train_output = trainer.train(resume_from_checkpoint=last_checkpoint) if last_checkpoint else trainer.train()\n    trainer.save_model(str(config.output_dir))\n    tokenizer.save_pretrained(str(config.output_dir))\n\n    validation_metrics = trainer.evaluate(eval_dataset=validation_dataset, metric_key_prefix="validation")\n    test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")\n\n    metrics = add_perplexity(\n        {\n            "config": {\n                **asdict(config),\n                "data_path": str(config.data_path),\n                "output_dir": str(config.output_dir),\n                "metrics_path": str(config.metrics_path),\n            },\n            "gpu": gpu_summary(),\n            "train_runtime_seconds": train_output.metrics.get("train_runtime"),\n            "train_steps_per_second": train_output.metrics.get("train_steps_per_second"),\n            "train_samples_per_second": train_output.metrics.get("train_samples_per_second"),\n            "train_loss": train_output.metrics.get("train_loss"),\n            "best_model_checkpoint": trainer.state.best_model_checkpoint,\n            "resumed_from_checkpoint": last_checkpoint,\n            **validation_metrics,\n            **test_metrics,\n        }\n    )\n    write_metrics(config.metrics_path, metrics)\n\n    print("Colab training complete.")\n    print(json.dumps(metrics, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote helper scripts.')


In [ ]:
import os

os.environ['FOUNDER_AI_COLAB_BASE_MODEL'] = 'Qwen/Qwen3-4B'
os.environ['FOUNDER_AI_COLAB_DATA_PATH'] = '/content/founderai-colab-v1/training_data/teranga_merged.jsonl'
os.environ['FOUNDER_AI_COLAB_OUTPUT_DIR'] = '/content/founderai-colab-v1/lora_adapter'
os.environ['FOUNDER_AI_COLAB_METRICS_PATH'] = '/content/founderai-colab-v1/lora_adapter/training_metrics.json'
os.environ['FOUNDER_AI_COLAB_USE_4BIT'] = 'true'
os.environ['FOUNDER_AI_COLAB_EPOCHS'] = '1'
os.environ['FOUNDER_AI_COLAB_MAX_SEQ_LENGTH'] = '512'
os.environ['FOUNDER_AI_COLAB_GRAD_ACCUM'] = '8'
os.environ['FOUNDER_AI_COLAB_SAVE_STEPS'] = '25'
os.environ['FOUNDER_AI_COLAB_EVAL_STEPS'] = '25'
os.environ['FOUNDER_AI_COLAB_SAMPLE_LIMIT'] = '120'
print('Training config ready. First run will use 120 samples for safety on Colab free.')


In [ ]:
!python /content/founderai-colab-v1/training_data/train_qwen3_lora_colab.py


In [ ]:
!ls -lah /content/founderai-colab-v1/lora_adapter


In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive('/content/founderai-colab-v1/founderai_lora_adapter', 'zip', '/content/founderai-colab-v1/lora_adapter')
print('Created archive:', archive_path)
files.download(archive_path)
